In [1]:
! pip install azure-eventhub requests

In [2]:
import json
import time
import requests
from azure.eventhub import EventData, EventHubProducerClient

# Fabric Eventstream / Event Hub credentials
CONNECTION_STR = (
    "Endpoint;"
    "SharedAccessKeyName;"
    "SharedAccessKey;"
    "EntityPath"
)
EVENT_HUB_NAME = "abc_eh"
OPEN_SKY_URL = "https://opensky-network.org/api/states/all"

# OpenSky Account Credentials 
OPENSKY_USER = "Your Username"              #Your Username
OPENSKY_PASSWORD = "Your Password"     #Your Password

def fetch_flight_data():
  try:
    # Pass auth tuple to prevent rate limiting 
    response = requests.get(
        OPEN_SKY_URL, auth=(OPENSKY_USER, OPENSKY_PASSWORD), timeout=10
    )
    if response.status_code == 200:
      return response.json()
    else:
      print(
          f"Error fetching OpenSky data: {response.status_code} -"
          f" {response.text}"
      )
      return None
  except Exception as e:
    print(f"Request failed: {e}")
    return None

def main():
  producer = EventHubProducerClient.from_connection_string(
      conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME
  )

  print(
      "Starting authenticated flight data poller with new credentials... Press"
      " Ctrl+C to stop."
  )
  with producer:
    while True:
      data = fetch_flight_data()
      if data and "states" in data and data["states"]:
        payload = {"Time": data.get("time"), "States": data.get("states")}
        json_data = json.dumps(payload)

        event_data_batch = producer.create_batch()
        event_data_batch.add(EventData(json_data))
        producer.send_batch(event_data_batch)

        print(
            f"Successfully sent payload with {len(data['states'])} flights to"
            " Fabric."
        )
      else:
        print("No flight data received or empty state array.")

      # Poll every 30 seconds to stay safely within OpenSky rate limits
      time.sleep(30)

if __name__ == "__main__":
  main()

Starting authenticated flight data poller with new credentials... Press Ctrl+C to stop.
Successfully sent payload with 6880 flights to Fabric.
Successfully sent payload with 6901 flights to Fabric.
Successfully sent payload with 6909 flights to Fabric.
Successfully sent payload with 6922 flights to Fabric.
Successfully sent payload with 6913 flights to Fabric.
Successfully sent payload with 6911 flights to Fabric.
Successfully sent payload with 6920 flights to Fabric.
Successfully sent payload with 6915 flights to Fabric.
Successfully sent payload with 6924 flights to Fabric.
Successfully sent payload with 6933 flights to Fabric.
Successfully sent payload with 6933 flights to Fabric.
Successfully sent payload with 6932 flights to Fabric.
Successfully sent payload with 6930 flights to Fabric.
Successfully sent payload with 6933 flights to Fabric.
Successfully sent payload with 6933 flights to Fabric.
Successfully sent payload with 6929 flights to Fabric.
Successfully sent payload with 6

KeyboardInterrupt: 